# ECDH-PSI + CKKS: обучение по всем признакам банка и телекома

**Цель.** Использовать исходные столбцы обеих таблиц, затем выбрать **до 100 признаков с помощью L1-регуляризации** на обучающей выборке и обучить CKKS-модель на всём train. Числовые признаки заполняются по медиане и масштабируются; категориальные — кодируются One-Hot. `SK_ID_CURR` — только идентификатор PSI, `TARGET` — только метка. Число признаков после One-Hot может быть **больше** числа исходных столбцов.

**Ключевые ограничения.** Это *исследовательский локальный прототип*, не безопасная двухсторонняя система. В одном процессе доступны оба датасета и секретный ключ; согласование строк выполняется по открытым ID после PSI. Расшифрование маскированных агрегатов **не защищает** от злоупотребления расшифровывающим сервисом. Используйте только общедоступные/синтетические данные.

**Ресурсы.** Без отбора несколько сотен столбцов × ~300 тыс. строк × много эпох CKKS требуют десятков/сотен тысяч шифрований и могут выполняться **очень долго**. Для выбранных признаков используется потоковое шифрование *по пакетам в каждую эпоху*, чтобы не хранить все шифротексты в ОЗУ. Сначала запустите короткий тест с `MAX_ROWS=4096`, `EPOCHS=1`, затем верните `MAX_ROWS=None`. Для экономии времени PSI можно сохранить в локальный кеш (кеш не является защищённым артефактом).

## 1. Установка и параметры

In [1]:
%pip -q install openmined-psi tenseal numpy pandas pyarrow scikit-learn scipy matplotlib

In [2]:
import time
import secrets
from pathlib import Path
import numpy as np
import pandas as pd
import scipy.sparse as sp
import tenseal as ts
import private_set_intersection.python as psi
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import SGDClassifier, LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, brier_score_loss,
    balanced_accuracy_score, accuracy_score, precision_score,
    recall_score, f1_score, confusion_matrix, precision_recall_curve,
)

SEED = 42
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
BATCH_SIZE = 1024   # <= 4096 при degree=8192; небольшой пакет снижает пиковую память
EPOCHS = 10         # 3 полных прохода HE; увеличивайте только после проверки
LEARNING_RATE = 0.15
REG_L2 = 1e-3
MAX_ROWS = None     # None = все доступные строки после PSI
MAX_FEATURES = 100  # Отбираем до 100 наиболее значимых признаков через L1
L1_C = 0.1        # Сила L1-регуляризации: меньше C = более сильный отбор
RUN_HE = True
RUN_PLAINTEXT_BASELINE = True
# PSI при повторном запуске можно прочитать из файла только для тестового Colab.
USE_PSI_CACHE = True
PSI_CACHE = Path('/content/psi_matched_ids_local_demo.npy')

print('Режим:', 'полная выборка' if MAX_ROWS is None else MAX_ROWS,
      '| Признаков после L1 не более:', MAX_FEATURES)

Режим: полная выборка | Признаков после L1 не более: 100


## 2. Загрузка таблиц и настоящий PSI по ID

In [3]:
locations = [
    (Path('/content/bank_psi.parquet'), Path('/content/telecom_train.parquet')),
    (Path('/content/data/bank_psi.parquet'), Path('/content/data/telecom_train.parquet')),
    (Path('data/bank_psi.parquet'), Path('data/telecom_train.parquet')),
]
source = next(((b, t) for b, t in locations if b.exists() and t.exists()), None)
if source is None:
    raise FileNotFoundError('Загрузите bank_psi.parquet и telecom_train.parquet в /content/.')

bank = pd.read_parquet(source[0])
telecom = pd.read_parquet(source[1])
for name, frame in [('банк', bank), ('телеком', telecom)]:
    if 'SK_ID_CURR' not in frame.columns:
        raise KeyError(f'В таблице «{name}» нет SK_ID_CURR')
    frame.dropna(subset=['SK_ID_CURR'], inplace=True)
    frame['SK_ID_CURR'] = frame['SK_ID_CURR'].astype('int64').astype(str)
    frame.drop_duplicates(subset=['SK_ID_CURR'], inplace=True)
    frame.reset_index(drop=True, inplace=True)
if 'TARGET' not in bank.columns:
    raise KeyError('В банке нет TARGET')

bank_ids = bank['SK_ID_CURR'].tolist()
tel_ids = telecom['SK_ID_CURR'].tolist()

# Привязка кеша к текущим наборам ID (предотвращает случайное использование старого пересечения).
import hashlib
fingerprint = hashlib.sha256(
    ('|'.join(bank_ids) + '#' + '|'.join(tel_ids)).encode('utf-8')
).hexdigest()[:16]
cache_file = PSI_CACHE.with_name(f'{PSI_CACHE.stem}_{fingerprint}.npy')

if USE_PSI_CACHE and cache_file.exists():
    matched_ids = np.load(cache_file, allow_pickle=False).astype(str).tolist()
    print('PSI: использован локальный кеш предыдущего запуска; криптография не повторялась.')
else:
    t0 = time.perf_counter()
    psi_client = psi.client.CreateWithNewKey(True)
    psi_server = psi.server.CreateWithNewKey(True)
    setup = psi_server.CreateSetupMessage(0.001, len(bank_ids), tel_ids, psi.DataStructure.RAW)
    request = psi_client.CreateRequest(bank_ids)
    response = psi_server.ProcessRequest(request)
    matched_idx = psi_client.GetIntersection(setup, response)
    matched_ids = [bank_ids[i] for i in matched_idx]
    print(f'ECDH-PSI: {len(matched_ids):,} совпадений за {time.perf_counter()-t0:.1f} с')
    if USE_PSI_CACHE:
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        np.save(cache_file, np.asarray(matched_ids, dtype='U32'), allow_pickle=False)

# ВАЖНО: это открытое согласование ID — только для локального эксперимента.
bank = bank.set_index('SK_ID_CURR').loc[matched_ids].reset_index()
telecom = telecom.set_index('SK_ID_CURR').loc[matched_ids].reset_index()
valid = bank['TARGET'].notna() & bank['TARGET'].isin([0, 1])
bank, telecom = bank.loc[valid].reset_index(drop=True), telecom.loc[valid].reset_index(drop=True)
if MAX_ROWS is not None and len(bank) > MAX_ROWS:
    _, chosen = train_test_split(
        np.arange(len(bank)), test_size=MAX_ROWS, random_state=SEED,
        stratify=bank['TARGET'].astype(int),
    )
    bank, telecom = bank.iloc[chosen].reset_index(drop=True), telecom.iloc[chosen].reset_index(drop=True)
assert bank['SK_ID_CURR'].equals(telecom['SK_ID_CURR'])
y = bank['TARGET'].astype('int8').to_numpy()
print(f'Клиентов: {len(y):,}; TARGET=1: {y.mean():.2%}')

PSI: использован локальный кеш предыдущего запуска; криптография не повторялась.
Клиентов: 307,491; TARGET=1: 8.07%


## 3. Подготовка всех признаков и L1-отбор только на train

Числовые и категориальные столбцы определяются автоматически. Одинаковые по имени признаки, встречающиеся в обеих таблицах, остаются **разными столбцами** с префиксами `bank__` и `telecom__`; это может дублировать информацию, если «телеком» построен из Home Credit. Константные/полностью пустые колонки, включая технические идентификаторы, исключаются. Для числовых значений используется медианная импутация, масштабирование и ограничение стандартизованных значений до ±5; категориальные превращаются в One-Hot. Импьютер и кодировщик обучаются **только на train**.

**Новый этап:** на train обучается L1-логистическая регрессия, которая зануляет часть коэффициентов. Берём до `MAX_FEATURES` признаков с ненулевыми коэффициентами и на validation/test используем точно те же столбцы. Это открытый **исследовательский** отбор; в реально разделённой системе ему потребуется конфиденциальная замена.


In [4]:
# Запрещённые поля: ID, целевая метка и очевидные производные целевой метки.
DROP_EXACT = {'SK_ID_CURR', 'TARGET'}
DROP_ID_COLUMNS = True

def select_columns(df):
    cols = [c for c in df.columns if c not in DROP_EXACT]
    if DROP_ID_COLUMNS:
        cols = [c for c in cols if not (
            c.upper().endswith('_ID') or c.upper().startswith('SK_ID_') or
            c.upper() in {'INDEX', 'ROW_ID'}
        )]
    return cols

bank_cols = select_columns(bank)
tel_cols = select_columns(telecom)
print('Исходные столбцы: банк=', len(bank_cols), 'телеком=', len(tel_cols))

X_bank_raw = bank[bank_cols].copy()
X_tel_raw = telecom[tel_cols].copy()
del bank, telecom

tr_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=SEED, stratify=y
)
train_idx, val_idx = train_test_split(
    tr_idx, test_size=VAL_SIZE_WITHIN_TRAIN, random_state=SEED, stratify=y[tr_idx]
)

# Всё, что не числовое, кодируем как категорию.
def clean_frame(frame):
    frame = frame.copy()
    for c in frame.columns:
        if pd.api.types.is_numeric_dtype(frame[c]):
            frame[c] = pd.to_numeric(frame[c], errors='coerce').replace([np.inf, -np.inf], np.nan)
        else:
            frame[c] = frame[c].astype('string').fillna('__MISSING__').astype(str)
    return frame

X_bank_raw = clean_frame(X_bank_raw)
X_tel_raw = clean_frame(X_tel_raw)

def make_preprocessor(frame):
    numeric = frame.select_dtypes(include=[np.number]).columns.tolist()
    categ = [c for c in frame.columns if c not in numeric]
    steps = []
    if numeric:
        steps.append(('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
        ]), numeric))
    if categ:
        steps.append(('categorical', Pipeline([
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=True, dtype=np.float32)),
        ]), categ))
    if not steps:
        raise ValueError('Нет признаков после отбора колонок')
    return ColumnTransformer(steps, sparse_threshold=1.0)

# Разные препроцессоры для двух сторон; fit только на train.
bank_transformer = make_preprocessor(X_bank_raw)
tel_transformer = make_preprocessor(X_tel_raw)

def fitted_split(transformer, df):
    train_data = transformer.fit_transform(df.iloc[train_idx])
    val_data = transformer.transform(df.iloc[val_idx])
    test_data = transformer.transform(df.iloc[test_idx])
    return [sp.csr_matrix(a, dtype=np.float32) for a in (train_data, val_data, test_data)]

B_train, B_val, B_test = fitted_split(bank_transformer, X_bank_raw)
T_train, T_val, T_test = fitted_split(tel_transformer, X_tel_raw)

# Удаляем промежуточные исходные таблицы — экономия памяти.
del X_bank_raw, X_tel_raw

# Чрезмерные выбросы в нормированных числовых признаках могут испортить
# приближение сигмоиды; ограничиваем только ненулевые элементы разреженных матриц.
def clip_sparse(m):
    m.data = np.clip(m.data, -5, 5)
    m.eliminate_zeros()
    return m

B_train, B_val, B_test = map(clip_sparse, [B_train, B_val, B_test])
T_train, T_val, T_test = map(clip_sparse, [T_train, T_val, T_test])

X_train = sp.hstack([B_train, T_train], format='csr', dtype=np.float32)
X_val = sp.hstack([B_val, T_val], format='csr', dtype=np.float32)
X_test = sp.hstack([B_test, T_test], format='csr', dtype=np.float32)

bank_names = ['bank__' + c for c in bank_transformer.get_feature_names_out()]
tel_names = ['telecom__' + c for c in tel_transformer.get_feature_names_out()]
all_feature_names = bank_names + tel_names

# L1-отбор выполняется исключительно на train, без использования validation и test.
# LogisticRegression с L1 зануляет коэффициенты малоинформативных признаков.
# Число признаков ограничивается MAX_FEATURES, но нулевые коэффициенты не добавляем.
original_features = X_train.shape[1]
if MAX_FEATURES is not None:
    if not isinstance(MAX_FEATURES, int) or MAX_FEATURES < 1:
        raise ValueError('MAX_FEATURES должен быть положительным целым числом или None')
    selector = LogisticRegression(
        penalty='l1', solver='liblinear', C=L1_C,
        max_iter=500, random_state=SEED,
    )
    selector.fit(X_train, y[train_idx].astype(int))
    importance = np.abs(selector.coef_[0])
    nonzero = np.flatnonzero(importance > 1e-10)
    if nonzero.size == 0:
        raise ValueError('L1 обнулил все признаки: увеличьте L1_C (например, до 1.0)')
    # При равной важности сортируем по индексу, чтобы результат был воспроизводим.
    selected_idx = sorted(nonzero, key=lambda j: (-importance[j], j))[:MAX_FEATURES]
    selected_idx = np.array(sorted(selected_idx), dtype=int)
    X_train = X_train[:, selected_idx].tocsr()
    X_val = X_val[:, selected_idx].tocsr()
    X_test = X_test[:, selected_idx].tocsr()
    all_feature_names = [all_feature_names[j] for j in selected_idx]
    print(f'L1: {original_features} → {len(selected_idx)} признаков; '
          f'ненулевых коэффициентов: {len(nonzero)}')
    print('По сторонам: банк =', sum(x.startswith('bank__') for x in all_feature_names),
          ', телеком =', sum(x.startswith('telecom__') for x in all_feature_names))
    display(pd.DataFrame({
        'Признак': all_feature_names,
        '|L1-коэффициент|': importance[selected_idx]
    }).sort_values('|L1-коэффициент|', ascending=False).head(20).reset_index(drop=True))
else:
    print(f'Отбор отключён: используются все {original_features} признаков')

P = X_train.shape[1]
y_train, y_val, y_test = y[train_idx].astype(float), y[val_idx].astype(float), y[test_idx].astype(float)
print(f'После One-Hot: {original_features} признаков; для CKKS выбрано {P}')
print(f'Train={len(y_train):,}, validation={len(y_val):,}, test={len(y_test):,}')
print(f'Доля TARGET=1: train={y_train.mean():.3%}, test={y_test.mean():.3%}')
print(f'Пакетов на эпоху: {(len(y_train)+BATCH_SIZE-1)//BATCH_SIZE:,}')
print(f'Шифрований признаков на эпоху ≈ {((len(y_train)+BATCH_SIZE-1)//BATCH_SIZE)*P:,}')

Исходные столбцы: банк= 380 телеком= 5
L1: 455 → 100 признаков; ненулевых коэффициентов: 318
По сторонам: банк = 99 , телеком = 1


,Признак,|L1-коэффициент|
0,bank__numeric__AMT_GOODS_PRICE,1.005542
1,bank__numeric__AMT_CREDIT,0.907594
2,bank__numeric__POS_SK_DPD_DEF_MAX,0.553820
3,bank__numeric__POS_SK_DPD_DEF_MEAN,0.456358
4,bank__numeric__INST_AMT_PAYMENT_MEAN,0.450251
5,bank__categorical__CODE_GENDER_F,0.419093
6,bank__numeric__INST_AMT_INSTALMENT_MEAN,0.387124
7,bank__numeric__INST_AMT_PAYMENT_MIN,0.343960
8,bank__numeric__BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,0.339044
9,bank__numeric__POS_CNT_INSTALMENT_MEAN,0.315890


После One-Hot: 455 признаков; для CKKS выбрано 100
Train=196,793, validation=49,199, test=61,499
Доля TARGET=1: train=8.073%, test=8.073%
Пакетов на эпоху: 193
Шифрований признаков на эпоху ≈ 19,300


## 4. CKKS: выбранные L1-признаки, потоковое шифрование и градиент

В каждом пакете **каждый обработанный признак** шифруется отдельно. Размер пакета ≤4096. Для экономии памяти все шифротексты не сохраняются: шифруется один пакет → вычисляется его вклад → переходим к следующему. Поэтому каждый проход повторно шифрует данные. Банк получает только шифротексты телеком-признаков **на уровне имитации**.

**Модель:** приближённый градиент логистической регрессии с \(\widetilde\sigma(z)=0.5+0.25z\). Это *не* точный градиент log-loss. Веса и bias обновляются открыто. Вычисление зашифрованного градиента содержит произведения шифротекстов, а значения финальных агрегатов раскрываются. Используется небезопасный учебный «сервис расшифрования», поэтому конфиденциальность всей схемы не гарантируется.

**Производительность:** сложность приблизительно линейна по числу признаков, клиентов и эпох. При сотнях/тысячах признаков полный запуск может потребовать очень много времени даже при небольшом числе эпох.

In [5]:
t0 = time.perf_counter()
private_context = ts.context(
    ts.SCHEME_TYPE.CKKS, poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[50, 30, 30, 30, 50]
)
private_context.global_scale = 2**30
private_context.generate_galois_keys()
private_context.generate_relin_keys()
public_context = ts.context_from(private_context.serialize(save_secret_key=False))
assert not public_context.has_secret_key()
print(f'Ключи CKKS созданы за {time.perf_counter()-t0:.1f} с; банк не владеет секретным ключом.')

rng_mask = secrets.SystemRandom()

def demo_decrypt_aggregate(encrypted_scalar):
    # Демонстрация, НЕ безопасный протокол расшифрования.
    mask = rng_mask.uniform(-16.0, 16.0)
    payload = (encrypted_scalar + mask).serialize()
    return float(ts.ckks_vector_from(private_context, payload).decrypt()[0]) - mask

def he_gradient(w, bias, X, labels, batch_size=BATCH_SIZE):
    p = X.shape[1]
    # Накопление только агрегатов по признакам: O(p) шифротекстов памяти.
    totals = [None] * (p + 1)
    for offset in range(0, X.shape[0], batch_size):
        block = X[offset:offset + batch_size].toarray()
        y_block = labels[offset:offset + batch_size].astype(float)
        # В реальном протоколе каждый владелец шифрует свою часть block.
        columns = [ts.ckks_vector(public_context, block[:, j].tolist()) for j in range(p)]
        encrypted_y = ts.ckks_vector(public_context, y_block.tolist())
        z = columns[0] * float(w[0])
        for j in range(1, p):
            z += columns[j] * float(w[j])
        z += float(bias)
        error = z * 0.25 + 0.5 - encrypted_y
        parts = [(error * col).sum() for col in columns]
        parts.append(error.sum())
        for j, part in enumerate(parts):
            totals[j] = part if totals[j] is None else totals[j] + part
        del block, columns, encrypted_y, z, error, parts
    sums = np.array([demo_decrypt_aggregate(x) for x in totals]) / X.shape[0]
    return sums[:-1], float(sums[-1])

def plain_gradient(w, bias, X, labels):
    e = 0.5 + 0.25 * (X @ w + bias) - labels
    return np.asarray(X.T @ e).ravel() / X.shape[0], float(np.mean(e))

def fit_he(X, labels, epochs=EPOCHS):
    w = np.zeros(X.shape[1], dtype=float)
    bias = float(4 * (labels.mean() - 0.5))
    history = []
    for epoch in range(epochs):
        tic = time.perf_counter()
        grad_w, grad_b = he_gradient(w, bias, X, labels)
        w -= LEARNING_RATE * (grad_w + REG_L2 * w)
        bias -= LEARNING_RATE * grad_b
        history.append({'epoch':epoch+1, 'seconds':time.perf_counter()-tic, 'grad_norm':np.linalg.norm(grad_w)})
        print(f'Эпоха {epoch+1}/{epochs}: {history[-1]["seconds"]:.1f} c; |grad|={history[-1]["grad_norm"]:.4f}')
    return w, bias, pd.DataFrame(history)

def fit_plain_approx(X, labels, epochs=EPOCHS):
    w = np.zeros(X.shape[1], dtype=float)
    bias = float(4 * (labels.mean() - 0.5))
    for _ in range(epochs):
        grad_w, grad_b = plain_gradient(w, bias, X, labels)
        w -= LEARNING_RATE * (grad_w + REG_L2 * w)
        bias -= LEARNING_RATE * grad_b
    return w, bias

Ключи CKKS созданы за 2.1 с; банк не владеет секретным ключом.


## 5. Сначала открытая модель: проверяем, помогают ли отобранные признаки

Открытый baseline `SGDClassifier(loss="log_loss")` — **отдельный алгоритм**, с настоящей логистической функцией потерь. Он не является эталоном арифметики CKKS. Он показывает, есть ли в признаках полезный сигнал до дорогого HE-эксперимента.

In [6]:
if RUN_PLAINTEXT_BASELINE:
    t0 = time.perf_counter()
    baseline = SGDClassifier(
        loss='log_loss', penalty='l2', alpha=1e-4,
        max_iter=40, tol=1e-3, random_state=SEED,
    )
    baseline.fit(X_train, y_train.astype(int))
    baseline_p = baseline.predict_proba(X_test)[:, 1]
    print(f'Открытая логистическая модель: {time.perf_counter()-t0:.1f} с')

Открытая логистическая модель: 2.5 с


## 6. HE-обучение и проверка точности CKKS

Одна проверка зашифрованного градиента на небольшом пакете. Затем обучение на **всех train-строках и отобранных L1-признаках**. Если `MAX_FEATURES = None`, L1-отбор отключён.

In [7]:
t0 = time.perf_counter()

m = min(BATCH_SIZE, len(y_train))
w0 = np.zeros(P)
b0 = float(4 * (y_train.mean() - 0.5))

g_he, gb_he = he_gradient(
    w0, b0,
    X_train[:m],
    y_train[:m]
)

batch_time = time.perf_counter() - t0
n_batches = int(np.ceil(len(y_train) / BATCH_SIZE))

estimated_seconds = batch_time * n_batches * EPOCHS

print('Признаков:', P)
print('Время одного пакета:', round(batch_time, 2), 'сек')
print('Пакетов на эпоху:', n_batches)
print('Эпох:', EPOCHS)
print('Прогноз времени:', round(estimated_seconds / 60, 1), 'мин')

Признаков: 100
Время одного пакета: 5.22 сек
Пакетов на эпоху: 193
Эпох: 10
Прогноз времени: 168.0 мин


In [8]:

import os
import json
import hashlib
import time
from pathlib import Path

from google.colab import drive

# Подключаем постоянное хранилище.
drive.mount('/content/drive')

CHECKPOINT_DIR = Path(
    '/content/drive/MyDrive/PSI_CKKS_checkpoints'
)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = CHECKPOINT_DIR / 'he_training.npz'


if RUN_HE:

    # ==========================================
    # 1. Идентификатор текущего эксперимента
    # ==========================================

    # Не загружаем старые веса, если изменились
    # признаки, выборка или параметры обучения.

    config = {
        'features': list(all_feature_names),
        'P': int(P),
        'n_train': int(len(y_train)),
        'batch_size': int(BATCH_SIZE),
        'learning_rate': float(LEARNING_RATE),
        'reg_l2': float(REG_L2),
    }

    fingerprint = hashlib.sha256(
        json.dumps(
            config,
            sort_keys=True
        ).encode('utf-8')
        + np.asarray(
            train_idx, dtype=np.int64
        ).tobytes()
        + np.asarray(
            y_train, dtype=np.float64
        ).tobytes()
    ).hexdigest()

    # ==========================================
    # 2. Загрузка сохранённого состояния
    # ==========================================

    w_he = np.zeros(P, dtype=float)
    b_he = float(4 * (y_train.mean() - 0.5))

    start_epoch = 0
    history = []

    if CHECKPOINT_FILE.exists():

        with np.load(
            CHECKPOINT_FILE,
            allow_pickle=False
        ) as saved:

            saved_fingerprint = str(
                saved['fingerprint'].item()
            )

            if saved_fingerprint != fingerprint:
                raise ValueError(
                    'Сохранённая модель относится '
                    'к другой выборке или другим '
                    'параметрам. Используй новый '
                    'файл контрольной точки.'
                )

            w_he = saved['weights'].astype(float)
            b_he = float(saved['bias'].item())
            start_epoch = int(saved['epoch'].item())

            history = json.loads(
                str(saved['history_json'].item())
            )

        print(
            'Загружена контрольная точка:',
            start_epoch,
            'завершённых эпох'
        )

    else:
        print('Контрольной точки нет: новое обучение.')

    # ==========================================
    # 3. Проверка начального HE-градиента
    # ==========================================

    if start_epoch == 0:

        t0 = time.perf_counter()

        m = min(BATCH_SIZE, len(y_train))

        g_he, gb_he = he_gradient(
            w_he,
            b_he,
            X_train[:m],
            y_train[:m]
        )

        g_plain, gb_plain = plain_gradient(
            w_he,
            b_he,
            X_train[:m],
            y_train[:m]
        )

        grad_error = np.max(
            np.abs(
                np.r_[
                    g_he - g_plain,
                    gb_he - gb_plain
                ]
            )
        )

        print(
            'Макс. погрешность градиента:',
            grad_error
        )

        np.testing.assert_allclose(
            np.r_[g_he, gb_he],
            np.r_[g_plain, gb_plain],
            atol=3e-3,
            rtol=0
        )

        print(
            'Тест градиента:',
            round(time.perf_counter() - t0, 1),
            'сек'
        )

    # ==========================================
    # 4. Обучение с сохранением после эпохи
    # ==========================================

    for epoch in range(start_epoch, EPOCHS):

        tic = time.perf_counter()

        print(
            f'\nНачало эпохи {epoch + 1}/{EPOCHS}',
            flush=True
        )

        # Зашифрованный градиент по всему train.
        grad_w, grad_b = he_gradient(
            w_he,
            b_he,
            X_train,
            y_train
        )

        # Те же обновления, что и в fit_he().
        w_he -= LEARNING_RATE * (
            grad_w + REG_L2 * w_he
        )

        b_he -= LEARNING_RATE * grad_b

        elapsed = time.perf_counter() - tic

        history.append({
            'epoch': epoch + 1,
            'seconds': elapsed,
            'grad_norm': float(
                np.linalg.norm(grad_w)
            )
        })

        # ======================================
        # 5. Сохранение на Google Drive
        # ======================================

        # Временный файл для безопасной замены.
        temp_file = CHECKPOINT_DIR / (
            'he_training.tmp.npz'
        )

        np.savez_compressed(
            temp_file,
            weights=w_he,
            bias=np.array(b_he),
            epoch=np.array(epoch + 1),
            fingerprint=np.array(fingerprint),
            history_json=np.array(
                json.dumps(history)
            )
        )

        os.replace(
            temp_file,
            CHECKPOINT_FILE
        )

        print(
            f'Эпоха {epoch + 1}/{EPOCHS}: '
            f'{elapsed:.1f} сек. '
            f'Состояние сохранено.',
            flush=True
        )

    # ==========================================
    # 6. Результаты CKKS
    # ==========================================

    history_he = pd.DataFrame(history)

    print('\n=== CKKS-обучение ===')
    print('Завершено эпох:', len(history))
    print('Всего эпох:', EPOCHS)

    if len(history) >= EPOCHS:
        print('Обучение завершено.')
    else:
        print('Обучение ещё не завершено.')

    display(history_he)

    # ==========================================
    # 7. Сравнение с открытым эталоном
    # ==========================================

    if len(history) >= EPOCHS:

        w_ref, b_ref = fit_plain_approx(
            X_train,
            y_train
        )

        coef_diff = np.max(
            np.abs(
                np.r_[
                    w_he - w_ref,
                    b_he - b_ref
                ]
            )
        )

        print(
            'Макс. отклонение весов и bias '
            'от открытого эталона:',
            coef_diff
        )

        print(
            'Контрольная точка:',
            CHECKPOINT_FILE
        )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ValueError: Сохранённая модель относится к другой выборке или другим параметрам. Используй новый файл контрольной точки.

## 7. Оценка на отложенном test: ROC-AUC, PR-AUC, Recall, Precision и Log-loss

Для измерения качества модели **не нужно раскрывать тестовые признаки через CKKS**: здесь используем тестовую матрицу только как локальный открытый эталон, а при желании ниже отдельно проверяем CKKS-предсказания одного пакета. Порог выбираем на **валидации**, а не на тесте. При сильном дисбалансе Accuracy сама по себе малоинформативна.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -30, 30)))

def best_f1_threshold(y_true, probabilities):
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    return float(thresholds[np.argmax(f1)])

def metric_row(y_true, probabilities, threshold):
    predicted = (probabilities >= threshold).astype(int)
    return {
        'ROC-AUC': roc_auc_score(y_true, probabilities),
        'PR-AUC (AP)': average_precision_score(y_true, probabilities),
        'Log-loss': log_loss(y_true, probabilities, labels=[0, 1]),
        'Brier': brier_score_loss(y_true, probabilities),
        'Accuracy': accuracy_score(y_true, predicted),
        'Balanced accuracy': balanced_accuracy_score(y_true, predicted),
        'Precision': precision_score(y_true, predicted, zero_division=0),
        'Recall': recall_score(y_true, predicted, zero_division=0),
        'F1': f1_score(y_true, predicted, zero_division=0),
    }

report = {}
prevalence = float(y_train.mean())
report['Без признаков'] = metric_row(y_test, np.full(len(y_test), prevalence), 0.5)

if RUN_PLAINTEXT_BASELINE:
    v = baseline.predict_proba(X_val)[:, 1]
    t = best_f1_threshold(y_val, v)
    report[f'Открытая лог. регрессия (порог {t:.3f})'] = metric_row(y_test, baseline_p, t)
    print('Порог открытой модели (val):', round(t, 4))

if RUN_HE:
    # Модель с приближённым градиентом; sigmoid для вывода вероятностей.
    he_val_p = sigmoid(X_val @ w_he + b_he)
    he_test_p = sigmoid(X_test @ w_he + b_he)
    he_cutoff = best_f1_threshold(y_val, he_val_p)
    report[f'HE, порог 0.5'] = metric_row(y_test, he_test_p, 0.5)
    report[f'HE, порог {he_cutoff:.3f}'] = metric_row(y_test, he_test_p, he_cutoff)
    print('Порог HE (val):', round(he_cutoff, 4))
    print('Матрица ошибок HE:')
    print(confusion_matrix(y_test, he_test_p >= he_cutoff, labels=[0, 1]))

display(pd.DataFrame(report).T.round(4))
print(f'Test: {len(y_test):,}, доля TARGET=1: {y_test.mean():.2%}')

## 8. Дополнительная CKKS-проверка прогнозов на одном тестовом пакете

Расшифрование индивидуальных логитов **раскрывает прогнозы банку**. Это только тест арифметики, не гарантия конфиденциальности предсказаний.

In [ ]:
if RUN_HE:
    block = X_test[:min(128, len(y_test))].toarray()
    encrypted_columns = [ts.ckks_vector(public_context, block[:, j].tolist()) for j in range(P)]
    enc_z = encrypted_columns[0] * float(w_he[0])
    for j in range(1, P):
        enc_z += encrypted_columns[j] * float(w_he[j])
    enc_z += float(b_he)
    # Векторная расшифровка здесь только для диагностики в локальной среде.
    he_p = sigmoid(np.asarray(ts.ckks_vector_from(private_context, enc_z.serialize()).decrypt()))
    plain_p = sigmoid(block @ w_he + b_he)
    print('Максимальная ошибка CKKS-прогнозов (128 клиентов):', np.max(np.abs(he_p - plain_p)))
    np.testing.assert_allclose(he_p, plain_p, atol=3e-3, rtol=0)

## Как читать результаты и что нужно улучшить

- **ROC-AUC:** качество ранжирования. 0.5 = случайный порядок, 1 = идеальный.
- **PR-AUC (Average Precision):** особенно полезна при редком `TARGET=1`; сравнивайте с долей дефолтов в test.
- **Recall:** доля найденных дефолтов; **Precision:** доля реальных дефолтов среди отмеченных моделью.
- **F1:** баланс Precision и Recall при выбранном на validation пороге.
- **Log-loss / Brier:** качество *вероятностей*. У модели с линейным приближением сигмоиды вероятности могут быть плохо откалиброваны даже при неплохой ROC-AUC.
- **Balanced accuracy:** средняя доля правильно распознанных клиентов обоих классов. Accuracy может быть высокой даже при Recall=0.

**Ограничения этого ноутбука:** открытая сборка строк после PSI, обе стороны в одном процессе, телеком-дешифратор без проверки запросов; совпадение по шифротекстам не доказывает защищённость протокола. Открытый baseline обучается с настоящей логистической потерей; HE — по приближённому градиенту. **Нельзя** считать их полностью одинаковыми моделями. Полный HE-эксперимент даже с 100 признаками может потребовать ресурсов намного больше, чем обычный Colab.

In [10]:
import numpy as np

path = '/content/drive/MyDrive/PSI_CKKS_checkpoints/he_training.npz'

with np.load(path, allow_pickle=False) as checkpoint:
    print('Содержимое файла:', checkpoint.files)

    for key in checkpoint.files:
        arr = checkpoint[key]
        print(key, '—', arr.shape)

Содержимое файла: ['weights', 'bias', 'epoch', 'fingerprint', 'history_json']
weights — (100,)
bias — ()
epoch — ()
fingerprint — ()
history_json — ()


In [11]:
import numpy as np
import json

path = '/content/drive/MyDrive/PSI_CKKS_checkpoints/he_training.npz'

with np.load(path, allow_pickle=False) as checkpoint:
    w_saved = checkpoint['weights'].copy()
    b_saved = float(checkpoint['bias'])
    epochs_saved = int(checkpoint['epoch'])
    history_saved = json.loads(str(checkpoint['history_json'].item()))

print('Сохранено эпох:', epochs_saved)
print('Количество коэффициентов:', len(w_saved))
print('Bias:', b_saved)

Сохранено эпох: 10
Количество коэффициентов: 100
Bias: -1.6757511571635955


In [12]:
required = [
    'B_train', 'B_val', 'B_test',
    'T_train', 'T_val', 'T_test',
    'all_feature_names',
    'bank_names',
    'y_train', 'y_val', 'y_test'
]

for name in required:
    print(
        name,
        '— есть' if name in globals() else '— отсутствует'
    )

B_train — есть
B_val — есть
B_test — есть
T_train — есть
T_val — есть
T_test — есть
all_feature_names — есть
bank_names — есть
y_train — есть
y_val — есть
y_test — есть


In [13]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# Банковские признаки, сохранившиеся после предыдущего L1-отбора
selected_bank_names = [
    name for name in all_feature_names
    if name.startswith('bank__')
]

# Соответствие названий банковских признаков их индексам
bank_name_to_idx = {
    name: i for i, name in enumerate(bank_names)
}

# Проверяем, что все признаки существуют в банковской матрице
missing = [
    name for name in selected_bank_names
    if name not in bank_name_to_idx
]

if missing:
    raise ValueError(f'Не найдены признаки: {missing[:5]}')

bank_idx = np.array([
    bank_name_to_idx[name]
    for name in selected_bank_names
])

# Модель 1: только банк
X_bank_train = B_train[:, bank_idx].tocsr()
X_bank_val = B_val[:, bank_idx].tocsr()
X_bank_test = B_test[:, bank_idx].tocsr()

# Модель 2: банк + все признаки телекома
X_joint_train = sp.hstack(
    [X_bank_train, T_train], format='csr'
)
X_joint_val = sp.hstack(
    [X_bank_val, T_val], format='csr'
)
X_joint_test = sp.hstack(
    [X_bank_test, T_test], format='csr'
)

print('=== СОСТАВ ПРИЗНАКОВ ===')
print('Банк:', X_bank_train.shape[1])
print('Телеком:', T_train.shape[1])
print('Банк + телеком:', X_joint_train.shape[1])

print('\nРазмер обучающей выборки:', len(y_train))
print('Размер тестовой выборки:', len(y_test))

=== СОСТАВ ПРИЗНАКОВ ===
Банк: 99
Телеком: 5
Банк + телеком: 104

Размер обучающей выборки: 196793
Размер тестовой выборки: 61499


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)
import time

def train_and_evaluate(X_tr, X_v, X_te, name):
    start = time.perf_counter()

    model = LogisticRegression(
        solver='liblinear',
        max_iter=300,
        random_state=SEED
    )

    model.fit(X_tr, y_train.astype(int))

    elapsed = time.perf_counter() - start

    # Выбираем порог по validation
    val_prob = model.predict_proba(X_v)[:, 1]

    thresholds = np.linspace(0.01, 0.99, 199)
    f1_values = [
        f1_score(y_val, val_prob >= t, zero_division=0)
        for t in thresholds
    ]
    threshold = float(thresholds[np.argmax(f1_values)])

    # Оценка на test
    prob = model.predict_proba(X_te)[:, 1]
    pred = (prob >= threshold).astype(int)

    result = {
        'Модель': name,
        'Признаков': X_tr.shape[1],
        'ROC-AUC': roc_auc_score(y_test, prob),
        'PR-AUC': average_precision_score(y_test, prob),
        'Log-loss': log_loss(y_test, prob),
        'Balanced Accuracy': balanced_accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, zero_division=0),
        'Recall': recall_score(y_test, pred, zero_division=0),
        'F1': f1_score(y_test, pred, zero_division=0),
        'Порог': threshold,
        'Время, с': elapsed
    }

    return model, result


bank_model, bank_result = train_and_evaluate(
    X_bank_train,
    X_bank_val,
    X_bank_test,
    'Только банк'
)

joint_model, joint_result = train_and_evaluate(
    X_joint_train,
    X_joint_val,
    X_joint_test,
    'Банк + весь телеком'
)

results = pd.DataFrame([
    bank_result,
    joint_result
])

display(results.round(4))

print('\nИзменение ROC-AUC:',
      round(joint_result['ROC-AUC'] - bank_result['ROC-AUC'], 4))

print('Изменение PR-AUC:',
      round(joint_result['PR-AUC'] - bank_result['PR-AUC'], 4))

,Модель,Признаков,ROC-AUC,PR-AUC,Log-loss,Balanced Accuracy,Precision,Recall,F1,Порог,"Время, с"
0,Только банк,99,0.7299,0.2067,0.2543,0.6484,0.1996,0.4580,0.2781,0.1238,23.9407
1,Банк + весь телеком,104,0.7336,0.2080,0.2535,0.6439,0.2126,0.4266,0.2838,0.1337,37.9455



Изменение ROC-AUC: 0.0037
Изменение PR-AUC: 0.0013


In [9]:

# ==========================================
# СРАВНЕНИЕ: БАНК vs БАНК + ТЕЛЕКОМ
# ==========================================

from sklearn.linear_model import LogisticRegression, SGDClassifier
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time

N_BANK_FEATURES = 100

# 1. L1-отбор только среди банковских признаков
start = time.perf_counter()

bank_selector = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=L1_C,
    max_iter=500,
    random_state=SEED
)

bank_selector.fit(B_train, y_train.astype(int))

importance = np.abs(bank_selector.coef_[0])
nonzero = np.flatnonzero(importance > 1e-10)

if len(nonzero) == 0:
    raise ValueError('L1 не выбрала ни одного банковского признака')

bank_selected_idx = np.array(
    sorted(
        sorted(nonzero, key=lambda j: (-importance[j], j))[
            :N_BANK_FEATURES
        ]
    ),
    dtype=int
)

# 2. Модель только с банковскими признаками
X_bank_train = B_train[:, bank_selected_idx].tocsr()
X_bank_val = B_val[:, bank_selected_idx].tocsr()
X_bank_test = B_test[:, bank_selected_idx].tocsr()

# 3. Модель с теми же банковскими признаками
#    и ВСЕМИ подготовленными телеком-признаками
X_joint_train = sp.hstack(
    [X_bank_train, T_train], format='csr'
)
X_joint_val = sp.hstack(
    [X_bank_val, T_val], format='csr'
)
X_joint_test = sp.hstack(
    [X_bank_test, T_test], format='csr'
)

print('=== Состав признаков ===')
print('Банк:', X_bank_train.shape[1])
print('Все признаки телекома:', T_train.shape[1])
print('Совместная модель:', X_joint_train.shape[1])

print('Train:', len(y_train))
print('Validation:', len(y_val))
print('Test:', len(y_test))

print(
    'Время отбора:',
    round(time.perf_counter() - start, 2),
    'сек'
)

# Названия выбранных банковских признаков
selected_bank_names = [
    bank_names[i] for i in bank_selected_idx
]

# Проверка возможного дублирования признаков
bank_base = {
    name.removeprefix('bank__')
    for name in selected_bank_names
}
tel_base = {
    name.removeprefix('telecom__')
    for name in tel_names
}

overlap = sorted(bank_base & tel_base)

print('Совпадающих названий признаков:', len(overlap))
if overlap:
    print('Примеры совпадений:', overlap[:10])


KeyboardInterrupt: 

In [ ]:

# ==========================================
# ОБУЧЕНИЕ ДВУХ МОДЕЛЕЙ
# ==========================================

def train_fast_model(X_tr, X_v, X_te, name):

    model = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1e-4,
        max_iter=40,
        tol=1e-3,
        random_state=SEED
    )

    start = time.perf_counter()
    model.fit(X_tr, y_train.astype(int))
    elapsed = time.perf_counter() - start

    # Вероятности на validation
    val_prob = model.predict_proba(X_v)[:, 1]

    # Порог выбираем только на validation
    threshold = best_f1_threshold(y_val, val_prob)

    # Оцениваем качество на test
    test_prob = model.predict_proba(X_te)[:, 1]

    metrics = metric_row(
        y_test, test_prob, threshold
    )

    metrics['Модель'] = name
    metrics['Признаков'] = X_tr.shape[1]
    metrics['Порог'] = threshold
    metrics['Время, с'] = elapsed

    return model, metrics


# 1. Только банк
bank_model, bank_metrics = train_fast_model(
    X_bank_train,
    X_bank_val,
    X_bank_test,
    'Только банк'
)

# 2. Банк + все признаки телекома
joint_model_all, joint_metrics = train_fast_model(
    X_joint_train,
    X_joint_val,
    X_joint_test,
    'Банк + весь телеком'
)

# 3. Сравнение
comparison = pd.DataFrame([
    bank_metrics,
    joint_metrics
]).set_index('Модель')

display(comparison.round(4))

print('\n=== Разница: совместная минус банковская ===')

for metric in ['ROC-AUC', 'PR-AUC (AP)', 'Recall', 'F1']:
    difference = (
        joint_metrics[metric] - bank_metrics[metric]
    )
    print(f'{metric}: {difference:+.4f}')

print(
    '\nСуммарное время обучения:',
    round(
        bank_metrics['Время, с']
        + joint_metrics['Время, с'],
        2
    ),
    'сек'
)
